## Predictions of Cityscapes-trained model

This is the first essential stage of the evaluation pipeline. Before evaluation, it is necessary to define a set of outputs that will allow a meaningful comparison between two model versions.

First, we run the Cityscapes-trained EoMT model on the entire Cityscapes validation dataset in order to get a set of semantic predictions.

The predictions are saved in .png format which is a standard semantic segmentation output format convenient for fair comparison with COCO-trained model.

*Inference code was taken and adapted from eomt/inference.ipynb*

In [1]:
import yaml
from lightning import seed_everything
import torch
import numpy as np
import warnings
from torch.nn import functional as F
from torch.amp.autocast_mode import autocast
import matplotlib.pyplot as plt
import importlib
from pathlib import Path
from tqdm import tqdm
import os
from PIL import Image


warnings.filterwarnings("ignore")

seed_everything(0, verbose=False)

device = 0

/home/elisa/outlierdrive/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
city_config_path = "../../eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"
trained_city_path = "../../eomt_checkpoints/eomt_cityscapes.bin"

data_path = "../../data/datasets"

output_city_path_png = "../../data/eomt_valset_predictions/cityscapes_model/png"

os.makedirs(output_city_path_png, exist_ok=True)

with open(city_config_path, "r") as f:
    city_config = yaml.safe_load(f)

---

## Load the data

In [5]:
import sys
sys.path.append("../../eomt")

In [6]:
data_module_name, class_name = city_config["data"]["class_path"].rsplit(".", 1)
data_module = getattr(importlib.import_module(data_module_name), class_name)
data_module_kwargs = city_config["data"].get("init_args", {})

data = data_module(
    path=data_path,
    batch_size=1,
    num_workers=0,
    check_empty_targets=False,
    **data_module_kwargs
).setup()

Explore what validation dataloader returns. Before running inference code, we need to inspect the structure of the batch and what it contains. The format of the batches is important because semantic segmentation datasets may return different batch structures (tuples, dictionaries etc).

By inspecting the structure of our data we ensure correct handling of images and target masks, which is crucial for the evaluation.

In [5]:
val_loader = data.val_dataloader()

In [6]:
batch = next(iter(val_loader))

print(type(batch))
print(len(batch))

<class 'tuple'>
2


In [7]:
for i, item in enumerate(batch):
    print(i, type(item))

0 <class 'tuple'>
1 <class 'tuple'>


As we can see from above outputs, batch contains two elements and both of them are tuples. At this point we can assume that batch[0] contains images and batch[1] contains targets. However, lets understand how exactly images and targets are organized inside each batch.

In [8]:
print("batch length:", len(batch))

for i, part in enumerate(batch):
    print(f"\nBatch part {i}: type={type(part)}, len={len(part)}")
    
    for j, item in enumerate(part):
        print(f"  item {j}: type={type(item)}")
        
        if hasattr(item, "shape"):
            print(f"    shape={item.shape}")
        elif isinstance(item, (list, tuple)):
            print(f"    len={len(item)}")
            if len(item) > 0:
                print(f"    first element type={type(item[0])}")
                if hasattr(item[0], "shape"):
                    print(f"    first element shape={item[0].shape}")

batch length: 2

Batch part 0: type=<class 'tuple'>, len=1
  item 0: type=<class 'torchvision.tv_tensors._image.Image'>
    shape=torch.Size([3, 1024, 2048])

Batch part 1: type=<class 'tuple'>, len=1
  item 0: type=<class 'dict'>


Now we know that the image is a TorchVision image tensor, its channel size and resolution. While target is not a mask as we expected before, but a dictionary. At this point, we have to explore the content of the dictionary in order to know the exact structure of the ground-truth annotations.

The format of the target is crucial for computing metrics and comparing predictions with ground truth.

In [9]:
target = batch[1][0]

print(target.keys())

for k, v in target.items():
    print(k, type(v))
    if hasattr(v, "shape"):
        print(" shape:", v.shape)
    else:
        print(" value:", v)

dict_keys(['masks', 'labels', 'is_crowd'])
masks <class 'torchvision.tv_tensors._mask.Mask'>
 shape: torch.Size([10, 1024, 2048])
labels <class 'torch.Tensor'>
 shape: torch.Size([10])
is_crowd <class 'torch.Tensor'>
 shape: torch.Size([10])


As the result shows, the dataset does not provide direct map of shape [H,W]. Instead, the annotations are stored in instance-type representation. For example:

masks
shape: [10, 1024, 2048]

This means that
- this image has 10 annotated objects
- each object has its own binary mask
- each mask has resolution 1024 × 2048

Labels store the semantic class associated with each mask.

is_crowd indicates whether a region is a crowd annotation or a grouped object. During evaluation is may be very important part to handle.

## Notice

Ground truth is not already semantic segmentation format. It means that before evaluation we must understand how to reconstruct the semantic target representation correctly.

Predictions usually produce semantic masks but the ground truth is instance-based annotated. So before computing metrics we might need some conversion.

## Manual search

In eomt/datasets/cityscapes_semantic.py we can see the function target_parser. It basically converts semantic ground truth into masks: [N, H, W], where each mask corresponds to one semantic class present in the image.

--- 

## Load the model

In [10]:

warnings.filterwarnings(
    "ignore",
    message=r".*Attribute 'network' is an instance of `nn\.Module` and is already saved during checkpointing.*",
)

# Load encoder
encoder_cfg = city_config["model"]["init_args"]["network"]["init_args"]["encoder"]
encoder_module_name, encoder_class_name = encoder_cfg["class_path"].rsplit(".", 1)
encoder_cls = getattr(importlib.import_module(encoder_module_name), encoder_class_name)
encoder = encoder_cls(img_size=data.img_size, **encoder_cfg.get("init_args", {}))

# Load network
network_cfg = city_config["model"]["init_args"]["network"]
network_module_name, network_class_name = network_cfg["class_path"].rsplit(".", 1)
network_cls = getattr(importlib.import_module(network_module_name), network_class_name)
network_kwargs = {k: v for k, v in network_cfg["init_args"].items() if k != "encoder"}

network = network_cls(
    masked_attn_enabled=False,
    num_classes=data.num_classes,
    encoder=encoder,
    **network_kwargs,
)

# Load Lightning module
lit_module_name, lit_class_name = city_config["model"]["class_path"].rsplit(".", 1)
lit_cls = getattr(importlib.import_module(lit_module_name), lit_class_name)
model_kwargs = {k: v for k, v in city_config["model"]["init_args"].items() if k != "network"}

if "stuff_classes" in city_config["data"].get("init_args", {}):
    model_kwargs["stuff_classes"] = city_config["data"]["init_args"]["stuff_classes"]

model = (
    lit_cls(
        img_size=data.img_size,
        num_classes=data.num_classes,
        network=network,
        **model_kwargs,
    )
    .eval()
    .to(device)
)

## Load weights

In [11]:
weights = torch.load(
    trained_city_path,
    map_location=f"cuda:{device}",
    weights_only=False
)

if "state_dict" in weights:
    weights = weights["state_dict"]

model.load_state_dict(weights, strict=False)

model.eval()

MaskClassificationSemantic(
  (network): EoMT(
    (encoder): ViT(
      (backbone): VisionTransformer(
        (patch_embed): PatchEmbed(
          (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
          (norm): Identity()
        )
        (pos_drop): Dropout(p=0.0, inplace=False)
        (patch_drop): Identity()
        (norm_pre): Identity()
        (blocks): Sequential(
          (0): Block(
            (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
            (attn): Attention(
              (qkv): Linear(in_features=768, out_features=2304, bias=True)
              (q_norm): Identity()
              (k_norm): Identity()
              (attn_drop): Dropout(p=0.0, inplace=False)
              (norm): Identity()
              (proj): Linear(in_features=768, out_features=768, bias=True)
              (proj_drop): Dropout(p=0.0, inplace=False)
            )
            (ls1): LayerScale()
            (drop_path1): Identity()
            (norm

In the output above we can see the exact architecture of the loaded semantic segmentation version of EoMT.

---

## Semantic inference

Here the target annotations are converted into semantic ground-truth map.

In [12]:
IGNORE_INDEX = 255

def infer_semantic(img, target):
    with torch.no_grad(), autocast(dtype=torch.float16, device_type="cuda"):
        imgs = [img.to(device)]
        img_sizes = [img.shape[-2:] for img in imgs]
        crops, origins = model.window_imgs_semantic(imgs)

        mask_logits_per_layer, class_logits_per_layer = model(crops)

        mask_logits = F.interpolate(
            mask_logits_per_layer[-1],
            data.img_size,
            mode="bilinear"
        )

        crop_logits = model.to_per_pixel_logits_semantic(
            mask_logits,
            class_logits_per_layer[-1]
        )

        logits = model.revert_window_logits_semantic(
            crop_logits,
            origins,
            img_sizes
        )

        preds = logits[0].argmax(0).cpu()

    pred_array = preds.numpy().astype(np.uint8)

    target_array = model.to_per_pixel_targets_semantic(
        [target],
        IGNORE_INDEX
    )[0].numpy().astype(np.uint8)

    return pred_array, target_array

Before running inference on the entire validation dataset, verify that predictions have expected semantic format and ground truth conversion is handled correctly. It is important that predictions and ground truth are compatible for evalution.

Firstly, run inference on one image.

In [ ]:
# retrive one image and corresponding annotation
img, target = data.val_dataloader().dataset[0]

In [14]:
# get predicted and ground truth semantic labels
pred_array, target_array = infer_semantic(img, target)

In [15]:
print(pred_array.shape, pred_array.dtype)
print(target_array.shape, target_array.dtype)
print(np.unique(pred_array))
print(np.unique(target_array))

(1024, 2048) uint8
(1024, 2048) uint8
[ 0  1  2  4  5  7  8  9 10 11 12 13]
[  0   1   2   4   5   7   8  10  11  13 255]


Here we ensure that **both predictions and ground truth** are semantic masks. This is excatly what we need for semantic segmentation evaluation pipeline.

--- 

Next code run on Colab (mainly I work on my local server) otherwise it will take very long time. It saves all the necessary predictions in the defined folder. I would like to save .png files with original names for easier analysis and later computations.

In [16]:
val_dataset = data.val_dataloader().dataset
print(dir(val_dataset))

['__add__', '__annotations__', '__class__', '__class_getitem__', '__del__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__len__', '__lt__', '__module__', '__ne__', '__new__', '__orig_bases__', '__parameters__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_load_zips', '_sort_key', 'close', 'imgs', 'is_crowd_by_id', 'labels_by_id', 'only_annotations_json', 'polygons_by_id', 'stuff_classes', 'target_folder_path_in_zip', 'target_instance_folder_path_in_zip', 'target_instance_zip', 'target_instance_zip_path', 'target_parser', 'target_zip', 'target_zip_path', 'target_zip_path_in_zip', 'targets', 'targets_instance', 'transforms', 'valid_member', 'zip', 'zip_path']


In [ ]:
val_dataset = data.val_dataloader().dataset

print("Validation set size:", len(val_dataset))

for idx in tqdm(range(len(val_dataset))):

    img, target = val_dataset[idx]
    pred_array, target_array = infer_semantic(img, target)

    original_path = val_dataset.imgs[idx]
    original_name = os.path.basename(original_path)

    base_name = original_name.replace("_leftImg8bit.png", "")
    png_name = f"{base_name}__predTrainIds.png"

    Image.fromarray(
        pred_array.astype(np.uint8)
    ).save(
        os.path.join(
            output_city_path_png,
            png_name
        )
    )

print("Finished saving PNG predictions")

The saved PNG masks contain integer class labels per pixel, so they are compatible with the converted ground-truth semantic masks. This step prepares the Cityscapes-trained model predictions in a standardized format, so later we can fairly compare it with predictions from the COCO-trained model.

---

## Check the predictions


In [ ]:
import os
import glob
import numpy as np
from PIL import Image

city_pred = "../../data/eomt_valset_predictions/cityscapes_model/png/content/drive/MyDrive/eomt_valset_predictions/cityscapes_model/png"

city_pngs = sorted(
    glob.glob(
        os.path.join(city_pred, "*_predTrainIds.png")
    )
)

print("Number of Cityscapes PNG predictions:", len(city_pngs))
print("First file:", city_pngs[0])

Number of Cityscapes PNG predictions: 500
First file: ../data/eomt_valset_predictions/cityscapes_model/png/content/drive/MyDrive/eomt_valset_predictions/cityscapes_model/png/frankfurt_000000_000294_predTrainIds.png


In [17]:
def load_mask(path):
    return np.array(Image.open(path), dtype=np.int64)

all_city_values = set()

for path in city_pngs:
    mask = load_mask(path)
    all_city_values.update(np.unique(mask).tolist())

print("All Cityscapes prediction values:")
print(sorted(all_city_values))

All Cityscapes prediction values:
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]


Here, during inference we do not restrict model to the overlapped class space. We will do it later in evaluation pipeline.
All 19 classes appear somewhere in the validation predictions.

---

## GT trainIDs convert

We convert ground truth labelIds to the corresponding trainIds, because we have to use them for COCO-trained model predictions and during the common evaluation.

**Notice:** we make manual conversion only to evaluate saved .png files against GT. In eomt_eval_iou.py this conversion is done automatically.

In [1]:
CITYSCAPES_ID_TO_TRAINID = {
    7: 0, 8: 1, 11: 2, 12: 3, 13: 4,
    17: 5, 19: 6, 20: 7, 21: 8, 22: 9,
    23: 10, 24: 11, 25: 12, 26: 13, 27: 14,
    28: 15, 31: 16, 32: 17, 33: 18
}

def label_ids_to_train_ids(label_ids):
    train_ids = np.full(label_ids.shape, IGNORE_INDEX, dtype=np.uint8)

    for label_id, train_id in CITYSCAPES_ID_TO_TRAINID.items():
        train_ids[label_ids == label_id] = train_id

    return train_ids

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

ground_truth = "../../data/datasets_unzip/gtFine/val"
gt_trainId = "../../data/datasets_unzip/gtFine_trainIds/val"


gt_labelid_paths = sorted(glob.glob(
    os.path.join(
        ground_truth,
        "*",
        "*_gtFine_labelIds.png"
    )
))

In [5]:
IGNORE_INDEX = 255


for gt_path in tqdm(gt_labelid_paths):
    city = os.path.basename(os.path.dirname(gt_path))

    os.makedirs(os.path.join(gt_trainId, city), exist_ok=True)

    label_ids = np.array(Image.open(gt_path), dtype=np.int64)
    train_ids = label_ids_to_train_ids(label_ids)

    out_name = os.path.basename(gt_path).replace(
        "_gtFine_labelIds.png",
        "_gtFine_labelTrainIds.png"
    )

    out_path = os.path.join(gt_trainId, city, out_name)

    Image.fromarray(train_ids).save(out_path)

print("Finished converting GT to trainIds")

100%|██████████| 500/500 [00:27<00:00, 18.18it/s]

Finished converting GT to trainIds


In [6]:
gt_train_paths = sorted(glob.glob(
    os.path.join(
        gt_trainId,
        "*",
        "*_gtFine_labelTrainIds.png"
    )
))

print("Number of converted GT masks:")
print(len(gt_train_paths))

Number of converted GT masks:
500


In [7]:
all_gt_values = set()

for path in gt_train_paths:

    mask = np.array(
        Image.open(path),
        dtype=np.int64
    )

    all_gt_values.update(
        np.unique(mask).tolist()
    )

print("All GT trainIds found:")
print(sorted(all_gt_values))

All GT trainIds found:
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 255]


## Summary

At this stage we perform semantic segmentation inference using the Cityscapes-trained EoMT model. Predictions were generated for the entire Cityscapes validation set and saved as PNG masks. 

We verified that the saved prediction masks use the Cityscapes **trainId**, which is the label space used for evaluation pipeline too.

Since the original Cityscapes ground-truth annotations are provided as **labelIds**, we also converted all ground-truth masks to the corresponding **trainIds**. This conversion is absolutely necessary because evaluation requires predictions and ground truth to be in the same label space. 